# 🩺 Agentic Medical Framework for Eczema
**Local RAG + Multi-Model Orchestration**  
Orchestrator: Gemma 4 E2B (local) | Specialist: MedGemma 1.5 4B (cloud) | KB: Hanifin & Rajka 1980

## Agent 1 — ClinicalExpertSystem (Backend/Tools)

In [ ]:
import json
import os
import re
import time
from dataclasses import dataclass, field
from typing import Optional

print('✅ Imports ready')

In [ ]:
class ClinicalExpertSystem:
    """
    Agent 1 — Backend tool provider for the Gemma 4 E2B orchestrator.
    Provides two primary tools:
      1. fetch_hr_definition  — Local RAG against knowledge_base_H&R.json
      2. consult_specialist   — Privacy-safe cloud call to MedGemma 1.5 4B
    """

    def __init__(self, kb_path: str = "knowledge_base_H&R.json"):
        self.kb_path = kb_path
        with open(kb_path, "r", encoding="utf-8") as f:
            self.kb = json.load(f)
        self.tool_call_log: list[dict] = []
        print(f"✅ Knowledge Base loaded — protocol: {self.kb.get('protocol_version')}")

    # ── Tool 1: Local RAG ────────────────────────────────────────────────────
    def fetch_hr_definition(self, criteria_type: str) -> dict:
        """
        Queries the local H&R knowledge base.
        Args:
            criteria_type: 'major' | 'minor' | 'grading_rule'
        Returns:
            Dict of criterion_key -> clinical_definition  (or rule string)
        """
        key_map = {
            "major": "major_criteria",
            "minor": "minor_criteria",
            "grading_rule": "grading_rule",
        }
        kb_key = key_map.get(criteria_type.lower())
        if not kb_key:
            result = {"error": f"Unknown criteria_type '{criteria_type}'. Use major/minor/grading_rule."}
        else:
            result = self.kb.get(kb_key, {})

        self.tool_call_log.append({
            "tool": "fetch_hr_definition",
            "input": criteria_type,
            "keys_returned": list(result.keys()) if isinstance(result, dict) else result,
        })
        return result

    # ── Tool 2: Cloud MedGemma Specialist ───────────────────────────────────
    def consult_specialist(
        self,
        image_crop_path: str,
        focus: str,
        medgemma_api_key: Optional[str] = None,
    ) -> dict:
        """
        Sends a locally-cropped ROI to Cloud MedGemma 1.5 4B.
        Privacy guarantee: only the ROI crop (no PII) is transmitted.
        Args:
            image_crop_path : path to the locally-segmented ROI image
            focus           : clinical focus area (e.g. 'flexural lichenification')
            medgemma_api_key: optional API key (reads MEDGEMMA_API_KEY env var if None)
        Returns:
            dict with 'status', 'findings', 'confidence'
        """
        api_key = medgemma_api_key or os.environ.get("MEDGEMMA_API_KEY", "<not-set>")

        if not os.path.exists(image_crop_path):
            result = {"status": "error", "findings": f"ROI file not found: {image_crop_path}"}
        else:
            # ── Simulated MedGemma API response ───────────────────────────
            # Replace the block below with a real requests.post() call when
            # the MedGemma 1.5 4B endpoint is available.
            # ------------------------------------------------------------------
            # import requests, base64
            # with open(image_crop_path, 'rb') as img:
            #     b64 = base64.b64encode(img.read()).decode()
            # resp = requests.post(
            #     'https://medgemma.googleapis.com/v1/analyze',
            #     headers={'Authorization': f'Bearer {api_key}'},
            #     json={'image_b64': b64, 'focus': focus}
            # )
            # result = resp.json()
            # ------------------------------------------------------------------
            time.sleep(0.3)  # simulate network latency
            result = {
                "status": "success",
                "focus": focus,
                "findings": (
                    f"[MedGemma 1.5 4B] ROI '{os.path.basename(image_crop_path)}' analysed for '{focus}'. "
                    f"Morphology: erythematous, scaly plaques with epidermal thickening consistent with "
                    f"lichenification. Satellite papules noted. High confidence for atopic pattern."
                ),
                "confidence": 0.91,
            }

        self.tool_call_log.append({
            "tool": "consult_specialist",
            "image_crop": image_crop_path,
            "focus": focus,
            "status": result.get("status"),
        })
        return result

    # ── Utility ─────────────────────────────────────────────────────────────
    def get_grading_rule(self) -> str:
        return self.kb.get("grading_rule", "")

    def dump_tool_log(self) -> list[dict]:
        return self.tool_call_log


# Quick smoke-test
ces = ClinicalExpertSystem()
print("Major keys :", list(ces.fetch_hr_definition('major').keys()))
print("Minor count:", len(ces.fetch_hr_definition('minor')))
print("Grading    :", ces.get_grading_rule())

## Agent 2 — Gemma 4 E2B Orchestrator
Implements the **Lookup-First** reasoning protocol and H&R scorer.

In [ ]:
def load_system_prompt(path: str = "system_prompt.txt") -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

SYSTEM_PROMPT = load_system_prompt()
print("✅ System prompt loaded —", len(SYSTEM_PROMPT), "chars")

In [ ]:
import re

# ── Tool-call parser ─────────────────────────────────────────────────────────
# Parses function-call stubs emitted inside <|think|> blocks by Gemma 4 E2B.
# Expected formats:
#   fetch_hr_definition("major")
#   consult_specialist("roi_crop.jpg", "flexural lichenification")

TOOL_PATTERN = re.compile(
    r'(fetch_hr_definition|consult_specialist)'
    r'\(\s*"([^"]+)"(?:\s*,\s*"([^"]+)")?\s*\)'
)

def dispatch_tool(match: re.Match, ces: ClinicalExpertSystem) -> str:
    """Execute the tool call and return a formatted string result."""
    fn, arg1, arg2 = match.group(1), match.group(2), match.group(3)
    if fn == "fetch_hr_definition":
        result = ces.fetch_hr_definition(arg1)
        return f"[KB:{arg1}] " + json.dumps(result, ensure_ascii=False)
    elif fn == "consult_specialist":
        result = ces.consult_specialist(arg1, arg2 or "general")
        return f"[MedGemma] " + json.dumps(result, ensure_ascii=False)
    return "[TOOL_ERROR] unknown function"

def resolve_tool_calls(think_text: str, ces: ClinicalExpertSystem) -> str:
    """Scan a <|think|> block, execute all tool calls, inject their results."""
    resolved = think_text
    for match in TOOL_PATTERN.finditer(think_text):
        result_str = dispatch_tool(match, ces)
        placeholder = f"\n  → TOOL_RESULT: {result_str}"
        resolved = resolved.replace(match.group(0), match.group(0) + placeholder, 1)
    return resolved

print("✅ Tool router ready")

In [ ]:
@dataclass
class HRScore:
    major_matched: list[str] = field(default_factory=list)
    minor_matched: list[str] = field(default_factory=list)

    @property
    def meets_criteria(self) -> bool:
        """Definitive diagnosis: ≥3 Major AND ≥3 Minor (H&R 1980)."""
        return len(self.major_matched) >= 3 and len(self.minor_matched) >= 3

    @property
    def verdict(self) -> str:
        if self.meets_criteria:
            return "✅ POSITIVE — Atopic Dermatitis (H&R 1980 criteria met)"
        return "❌ NEGATIVE — Insufficient criteria for definitive diagnosis"

    def summary(self) -> str:
        lines = [
            f"H&R Score  : {len(self.major_matched)}/4 Major | {len(self.minor_matched)}/23 Minor",
            f"Verdict    : {self.verdict}",
            f"Major      : {self.major_matched}",
            f"Minor      : {self.minor_matched}",
        ]
        return "\n".join(lines)


def calculate_hr_score(
    verified_major: list[str],
    verified_minor: list[str],
    ces: ClinicalExpertSystem,
) -> HRScore:
    """
    Validates supplied criteria keys against the KB and returns an HRScore.
    Raises ValueError if any key is not found in the KB (prevents hallucination).
    """
    kb_major = ces.fetch_hr_definition("major")
    kb_minor = ces.fetch_hr_definition("minor")

    for key in verified_major:
        if key not in kb_major:
            raise ValueError(f"Major criterion '{key}' not in KB — cannot score.")
    for key in verified_minor:
        if key not in kb_minor:
            raise ValueError(f"Minor criterion '{key}' not in KB — cannot score.")

    return HRScore(major_matched=verified_major, minor_matched=verified_minor)


print("✅ H&R Scorer ready")

In [ ]:
def run_diagnostic_session(
    patient_context: str,
    roi_crop_path: str,
    ces: ClinicalExpertSystem,
    verified_major: list[str],
    verified_minor: list[str],
) -> dict:
    """
    Simulates a full Gemma 4 E2B reasoning session.

    In production this function passes SYSTEM_PROMPT + patient_context to the
    ML Kit GenAI Prompt API (4-bit KV-cache quantised Gemma 4 E2B), streams the
    <|think|> block token-by-token, intercepts tool calls, injects results,
    and returns the final report.

    For now it executes the 5-step protocol deterministically so every
    component can be tested offline.
    """
    print("="*60)
    print("GEMMA 4 E2B — DIAGNOSTIC SESSION")
    print("="*60)

    # ── Step 1 · Lookup ───────────────────────────────────────────────────────
    print("\n<|think|>")
    print("Step 1 · Lookup — fetching H&R definitions from KB ...")
    major_defs = ces.fetch_hr_definition("major")
    minor_defs = ces.fetch_hr_definition("minor")
    grading    = ces.get_grading_rule()
    print(f"  → Loaded {len(major_defs)} Major, {len(minor_defs)} Minor criteria.")
    print(f"  → Grading rule: {grading}")

    # ── Step 2 · Consult ──────────────────────────────────────────────────────
    print("\nStep 2 · Consult — dispatching ROI to MedGemma 1.5 4B ...")
    specialist_report = ces.consult_specialist(roi_crop_path, "flexural lichenification")
    print(f"  → {specialist_report.get('findings', 'No findings')}")
    print(f"  → Confidence: {specialist_report.get('confidence', 'N/A')}")

    # ── Step 3 · Verify ───────────────────────────────────────────────────────
    print("\nStep 3 · Verify — cross-referencing inputs against KB definitions ...")
    for key in verified_major:
        defn = major_defs.get(key, "NOT FOUND")
        print(f"  ✔ MAJOR '{key}': {defn[:80]}...")
    for key in verified_minor:
        defn = minor_defs.get(key, "NOT FOUND")
        print(f"  ✔ MINOR '{key}': {defn[:80]}...")

    # ── Step 4 · Calculate ────────────────────────────────────────────────────
    print("\nStep 4 · Calculate — scoring against H&R 1980 grading rule ...")
    score = calculate_hr_score(verified_major, verified_minor, ces)
    print(f"  → {score.summary()}")

    # ── Step 5 · Constraint Enforcement ──────────────────────────────────────
    print("\nStep 5 · Constraint Enforcement — verifying KB-only sourcing ...")
    for entry in ces.dump_tool_log():
        print(f"  → {entry}")
    print("  ✅ All criteria sourced from KB. Internal memory not used.")
    print("</|think|>")

    # ── Final Report ──────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("CLINICAL ASSESSMENT")
    print("="*60)
    print(f"Patient context : {patient_context}")
    print(f"Specialist input: {specialist_report.get('findings', '')}")
    print()
    print(score.summary())
    print()
    print("Source: knowledge_base_H&R.json | protocol_version:", ces.kb.get('protocol_version'))

    return {
        "score": score,
        "specialist_report": specialist_report,
        "tool_log": ces.dump_tool_log(),
    }


print("✅ Orchestrator ready")

## Demo Run — End-to-End Diagnostic Session

In [ ]:
import os, pathlib

# Create a dummy ROI crop file for the demo
DEMO_CROP = "demo_roi_crop.jpg"
pathlib.Path(DEMO_CROP).write_bytes(b"\xff\xd8\xff" + b"\x00" * 512)  # minimal JPEG stub

# Clinician-supplied criteria keys (would come from the Kotlin UI in production)
DEMO_MAJOR = ["pruritus", "distribution", "chronic_history"]
DEMO_MINOR = ["xerosis", "elevated_ige", "early_age_onset"]

result = run_diagnostic_session(
    patient_context="28-year-old male, antecubital fossa rash, 3-year history, known hay fever.",
    roi_crop_path=DEMO_CROP,
    ces=ces,
    verified_major=DEMO_MAJOR,
    verified_minor=DEMO_MINOR,
)